## Zusammenfassung

Ein Logistik-Betriebsteam plant einen randomisierten Feldversuch, der drei Fahrer-Routing-Strategien (statische Baseline, dynamische Umleitung, KI-optimiert) über drei Depot-Regionen hinweg vergleicht, mit der durchschnittlichen Lieferverzögerung (in Minuten) als Zielgröße. PROC GLMPOWER nimmt einen *exemplarischen* Datensatz vermuteter Zellmittelwerte und löst nach der Gesamtzahl der benötigten Fahrer auf, um Strategieeffekte bei 80 % und 90 % Power zu erkennen, und bildet dann ab, wie die erreichbare Power abnimmt, wenn die Streuung zwischen Routen wächst.

# Dimensionierung eines Fahrer-Routing-Feldversuchs mit PROC GLMPOWER

## Zusammenfassung

Ein Logistik-Betriebsteam steht kurz davor, einen randomisierten Feldversuch zu starten, der drei Fahrer-Routing-Strategien vergleicht -- eine **statische** Baseline, ein **dynamisches** Umleitungssystem und einen **KI-optimierten** Planer -- eingesetzt über drei Depot-Regionen (Northeast, Midwest, West). Die Zielgröße ist die durchschnittliche **Lieferverzögerung in Minuten**. Bevor Flottenkapazität für den Versuch gebunden wird, muss das Team beantworten: *Wie viele Fahrer benötigen wir, um die erwartete betriebliche Verbesserung zuverlässig zu erkennen?*

Dieses Notebook nutzt **PROC GLMPOWER**, um eine prospektive Power- und Stichprobenumfangsanalyse für das allgemeine lineare Modell hinter dem Versuch durchzuführen. Ausgehend von einem *exemplarischen* Datensatz vermuteter Zellmittelwerte wird (1) nach der gesamten Einschreibungszahl aufgelöst, die für 80 % und 90 % Power bei den Omnibus-Effekten Strategie und Region nötig ist, (2) abgebildet, wie die erreichbare Power abnimmt, wenn die Streuung zwischen Routen steigt, und (3) eine Power-Kurve erzeugt, die die Einschreibungsentscheidung unterstützt.

> **Kernaussage:** Bei einer plausiblen Fehler-Standardabweichung von 8 Minuten liefern rund **63 Fahrer** 80 % Power und **83 Fahrer** 90 % Power zur Erkennung der Routing-Strategie-Effekte -- aber wenn die Verzögerungsstreuung eher bei 10 Minuten liegt, erreichen selbst 90 eingeschriebene Fahrer keine 70 % Power, was den Wert enger, gut instrumentierter Routen unterstreicht.

---
## 1. Aufbau des exemplarischen Designs

Der erste Schritt kodiert das Versuchslayout und die vom Team *vermutete* mittlere Verzögerung für jede Kombination aus Routing-Strategie und Depot-Region. Wir legen einen festen Zufallssaat fest und fügen einen vernachlässigbaren Jitter hinzu, damit die Mittelwerte realistisch wirken, während die ausgeglichene 3x3-Struktur erhalten bleibt. Das `cell_n`-Gewicht von 1 in jeder Zelle teilt GLMPOWER mit, dass das Design ausgeglichen ist.

In [1]:
/* ================================================================
   Erzeugung des exemplarischen Datensatzes vermuteter mittlerer
   Verzoegerungen. Eine Zeile pro Design-Zelle
   Routing-Strategie x Depot-Region.
   ================================================================ */
DATEN routing_trial;
   AUFRUFEN streaminit(20260531);
   LÄNGE routing_strategy $8 depot_region $9;
   FELD strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   FELD region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   FELD smean[3]     _temporary_ (18.0 14.5 11.0);   /* vermutete Strategiemittelwerte */
   FELD radj[3]      _temporary_ (1.5  0.0 -1.0);    /* regionale Anpassungen (Min.)   */
   AUSFÜHRUNG i = 1 BIS 3;
      AUSFÜHRUNG j = 1 BIS 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         AUSGABE;
      ENDE;
   ENDE;
   ENTFERNEN i j;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=routing_trial BEZEICHNUNG noobs;
   VAR routing_strategy depot_region mean_delay_min cell_n;
   BEZEICHNUNG routing_strategy="Routing-Strategie" depot_region="Depot-Region"
         mean_delay_min="Mittlere Lieferverzögerung (Min.)" cell_n="Zellgewicht";
   TITEL "Exemplarische Zellmittelwerte: Vermutete Lieferverzögerung (Minuten)";
AUSFÜHREN;


                          Exemplarische Zellmittelwerte: Vermutete Lieferverzögerung (Minuten)                          

Routing-Strategie  Depot-Region   Mittlere Lieferverzögerung (Min.)  Zellgewicht
Static             Northeast                           19.687507651            1
Static             Midwest                            17.9871067398            1
Static             West                               16.8240210086            1
Dynamic            Northeast                          15.9537535365            1
Dynamic            Midwest                            14.4415169858            1
Dynamic            West                               12.8636389493            1
AIOpt              Northeast                          12.6143811724            1
AIOpt              Midwest                             10.893885771            1
AIOpt              West                                 9.635351403            1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Stichprobenumfang für das Omnibus-Design

Mit dem fertigen Design bitten wir GLMPOWER, den **Gesamtstichprobenumfang zu lösen** (`NTOTAL = .`) bei 80 % und 90 % Power. Die `MODEL`-Anweisung spezifiziert ein zweifaktorielles Layout mit Interaktion (`routing_strategy*depot_region`), `WEIGHT cell_n` deklariert die ausgeglichenen Profilgewichte, und `STDDEV = 8` ist der angenommene mittlere quadratische Fehler der Lieferverzögerung. `NFRACTIONAL` lässt die Prozedur exakte gebrochene Zahlen vor dem Aufrunden ausgeben.

Wir registrieren außerdem drei geplante `CONTRAST`-Vergleiche im Voraus -- KI-optimiert vs. Static, Dynamic vs. Static und jede Umleitung vs. Static --, die die operational bedeutsamen Hypothesen dokumentieren, für deren Prüfung der Versuch aufgebaut ist.

In [2]:
/* ================================================================
   Aufloesen nach der Anzahl der benoetigten Fahrer, um 80 % und 90 %
   Power fuer die Effekte Routing-Strategie und Depot-Region zu erreichen.
   ================================================================ */
PROZEDUR glmpower DATEN=routing_trial;
   KLASSE routing_strategy depot_region;
   MODELL mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   GEWICHT cell_n;
   BEZEICHNUNG routing_strategy="Routing-Strategie" depot_region="Depot-Region"
         mean_delay_min="Mittlere Lieferverzögerung (Min.)";
   CONTRAST "KI-optimiert vs. Static"  routing_strategy -1  0  1;
   CONTRAST "Dynamic vs. Static"       routing_strategy -1  1  0;
   CONTRAST "Jede Umleitung vs. Static" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITEL "Stichprobenumfang zur Erkennung der Routing-Strategie-Effekte auf die Verzögerung";
AUSFÜHREN;


                          Exemplarische Zellmittelwerte: Vermutete Lieferverzögerung (Minuten)                          

The GLMPOWER Procedure


               Fixed Scenario Elements                

Item                Value                             
------------------  ----------------------------------
Dependent Variable  Mittlere Lieferverzögerung (Min.) 
Source              Routing-Strategie                 
Source              Depot-Region                      
Source              Routing-Strategie*Depot-Region    

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Power-Sensitivität gegenüber Streuung und Versuchsgröße

Die Antwort zum Stichprobenumfang hängt von der angenommenen Fehler-Standardabweichung ab, die im Voraus selten genau bekannt ist. Hier drehen wir die Frage um: Wir **fixieren** mehrere Kandidaten für die Gesamteinschreibung (`NTOTAL = 45 90 180`) und **lösen nach der erreichten Power** (`POWER = .`) über ein Raster plausibler Verzögerungs-Standardabweichungen (6, 8 und 10 Minuten). Die resultierende Tabelle ist eine Sensitivitätskarte -- sie zeigt, wie robust jeder Einschreibungsplan ist, falls die reale Routen-Streuung höher ausfällt als erhofft.

In [3]:
/* ================================================================
   Sensitivitaetsraster: erreichte Power ueber Kandidaten-Versuchsgroessen
   und plausible Verzoegerungs-Standardabweichungen.
   ================================================================ */
PROZEDUR glmpower DATEN=routing_trial;
   KLASSE routing_strategy depot_region;
   MODELL mean_delay_min = routing_strategy depot_region;
   GEWICHT cell_n;
   BEZEICHNUNG routing_strategy="Routing-Strategie" depot_region="Depot-Region"
         mean_delay_min="Mittlere Lieferverzögerung (Min.)";
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITEL "Erreichte Power über Streuungs- und Versuchsgrößen-Szenarien";
AUSFÜHREN;


                          Exemplarische Zellmittelwerte: Vermutete Lieferverzögerung (Minuten)                          

The GLMPOWER Procedure


               Fixed Scenario Elements                

Item                Value                             
------------------  ----------------------------------
Dependent Variable  Mittlere Lieferverzögerung (Min.) 
Source              Routing-Strategie                 
Source              Depot-Region                      

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Power-Kurve für die Einschreibungsentscheidung

Zuletzt zeichnen wir eine **Power-Kurve** -- erreichte Power, während die Gesamteinschreibung von 30 auf 270 Fahrer in Schritten von 30 wächst -- für das Strategie-plus-Region-Modell bei der erwarteten Standardabweichung von 8 Minuten. Das Lösen von `POWER = .` über dieses Einschreibungsraster erzeugt die Kurve als tabellierte Reihe `N Total` vs. `Power`, sodass wir ablesen können, bei welcher Einschreibung jedes der konventionellen Ziele von 80 % und 90 % überschritten wird.

In [4]:
/* ================================================================
   Power-Kurve: erreichte Power vs. Gesamtzahl eingeschriebener Fahrer,
   von 30 bis 270 in Schritten von 30 bei der erwarteten Standard-
   abweichung von 8 Minuten.
   ================================================================ */
PROZEDUR glmpower DATEN=routing_trial;
   KLASSE routing_strategy depot_region;
   MODELL mean_delay_min = routing_strategy depot_region;
   GEWICHT cell_n;
   BEZEICHNUNG routing_strategy="Routing-Strategie" depot_region="Depot-Region"
         mean_delay_min="Mittlere Lieferverzögerung (Min.)";
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITEL "Power-Kurve: Erreichte Power vs. Gesamtzahl eingeschriebener Fahrer";
AUSFÜHREN;


                          Exemplarische Zellmittelwerte: Vermutete Lieferverzögerung (Minuten)                          

The GLMPOWER Procedure


               Fixed Scenario Elements                

Item                Value                             
------------------  ----------------------------------
Dependent Variable  Mittlere Lieferverzögerung (Min.) 
Source              Routing-Strategie                 
Source              Depot-Region                      

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interpretation und Empfehlung

Die Analyse gibt dem Betriebsteam einen belastbaren Einschreibungsplan:

- **Basis-Dimensionierung (Abschnitt 2).** Unter Annahme einer Rest-Standardabweichung von 8 Minuten erreicht das volle zweifaktorielle Modell (Strategie, Region und deren Interaktion) **80 % Power bei 63 Fahrern** und **90 % Power bei 83 Fahrern**. Aufgerundet für Attrition liegt ein Einschreibungsziel nahe **90 Fahrern** komfortabel über der 90-%-Schwelle.

- **Sensitivität ist entscheidend (Abschnitt 3).** Die Power reagiert stark auf die Verzögerungsstreuung. Bei 90 Fahrern fällt die erreichte Power von ~99 % (SD=6) auf ~87 % (SD=8) auf ~68 % (SD=10). Ein Pilotversuch mit 45 Fahrern ist nur angemessen, wenn die Streuung gering ist (~81 % Power bei SD=6), und ist deutlich unterdimensioniert (~37 %), wenn SD 10 erreicht. Die praktische Implikation: In konsistente, gut instrumentierte Routen zu investieren, um die Streuung niedrig zu halten, ist ebenso wertvoll wie zusätzliche Fahrer.

- **Die Power-Kurve (Abschnitt 4).** Für das Strategie-plus-Region-Modell (ohne Interaktionsterm, die Sichtweise, die für den Sensitivitäts-Sweep verwendet wurde) gezeichnet, steigt die erreichte Power von 0,37 bei 30 Fahrern auf 0,69 bei 60, 0,87 bei 90 und 0,95 bei 120, und flacht oberhalb von 0,99 bei 180 ab. Liest man die Kurve gegen die Zielwerte, wird 80 % Power bei nahe 77 Fahrern erreicht und 90 % bei nahe 99 -- etwas höher als die Dimensionierung des vollen Modells in Abschnitt 2, weil das Weglassen des Interaktionsterms denselben Effekt auf weniger Modell-Freiheitsgrade verteilt.

**Empfehlung:** Rund **90 Fahrer** einschreiben (30 pro Routing-Strategie, ausgeglichen über die drei Depot-Regionen). Dies erreicht 90 % Power für das volle Modell unter der erwarteten Standardabweichung von 8 Minuten, hält ~87 % Power selbst auf der konservativeren reduzierten-Modell-Kurve, und hält den Versuch klein genug, um innerhalb eines einzigen Betriebsquartals durchgeführt zu werden.

*Hinweis:* GLMPOWER analysiert das vermutete *Design*, nicht beobachtete Ergebnisse -- daher hängt die Glaubwürdigkeit dieser Zahlen von den vermuteten Mittelwerten und der Standardabweichung ab. Teams sollten die Dimensionierung überarbeiten, sobald ein kurzer Pilotversuch eine empirische Schätzung der Lieferverzögerungs-Streuung liefert.